# E-Mail Classifier Using Hugging Face Model
Here we use Hugging face pretrained model. We do NOT define these classes in advance.


**You provide candidate labels at inference time.**

I used GPU on COLAB.

In [1]:
# check package version
import importlib.metadata

# List the distribution package names
packages = ["transformers", "torch"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")

# transformers version: 5.16.1
# torch version: 2.11.0+cu128

transformers version: 5.16.1
torch version: 2.11.0+cu128


In [2]:
# NOTE: First time it took 1-2 minutes

import time
import torch
from transformers import pipeline

# 1. Automatically detect and leverage your laptop's GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (CUDA)' if device == 0 else 'CPU'}\n")

# 2. Initialize the Zero-Shot pipeline
print("Loading model (this takes a few seconds the first time)...")

# This is 1.63GB file
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=device
)

print("Model loaded successfully!\n" + "="*50)

Using device: GPU (CUDA)

Loading model (this takes a few seconds the first time)...


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
# give email, categories and the prints the category with confidence score

def run_demo(email_text, categories):
    print(f"\n📧 Input Email:\n{email_text.strip()}")
    print(f"\n🏷️ Candidate Categories: {categories}")

    # Time the performance to show off GPU speed
    start_time = time.time()
    result = classifier(email_text, categories)
    end_time = time.time()

    print(f"⏱️ Inference Time: {end_time - start_time:.4f} seconds")
    print("\n📊 Classification Results:")

    # Print sorted results
    for label, score in zip(result['labels'], result['scores']):
        print(f"  - {label}: {score*100:.1f}%")
    print("="*50)


In [4]:

# --- LIVE DEMO SCENARIOS ---

# Scenario A: Standard Support Routing
email_1 = """
Hello, I am writing to request a full refund for my order #10293.
The software keeps crashing every time I click the export button, and I cannot use it for work.
"""
categories_1 = ["Refund Request", "Technical Support", "Sales Inquiry", "Spam"]
run_demo(email_1, categories_1)



📧 Input Email:
Hello, I am writing to request a full refund for my order #10293. 
The software keeps crashing every time I click the export button, and I cannot use it for work.

🏷️ Candidate Categories: ['Refund Request', 'Technical Support', 'Sales Inquiry', 'Spam']
⏱️ Inference Time: 1.0804 seconds

📊 Classification Results:
  - Refund Request: 67.2%
  - Spam: 18.5%
  - Technical Support: 7.4%
  - Sales Inquiry: 6.9%


In [5]:
# Scenario B: Dynamically changing categories live in class

email_2 = """
Hey! Check out these amazing discounts on premium leather jackets!
Click this link now to get 50% off before midnight!!!
"""
# You can show the class how changing this list changes the model's objective on the fly
categories_2 = ["Urgent Business", "Marketing/Spam", "Personal Chat"]
run_demo(email_2, categories_2)



📧 Input Email:
Hey! Check out these amazing discounts on premium leather jackets! 
Click this link now to get 50% off before midnight!!!

🏷️ Candidate Categories: ['Urgent Business', 'Marketing/Spam', 'Personal Chat']
⏱️ Inference Time: 0.1067 seconds

📊 Classification Results:
  - Marketing/Spam: 40.4%
  - Urgent Business: 31.5%
  - Personal Chat: 28.1%
